
# EX: Validating Heuristics

Before deploying an $A^*$ routing agent, the AI Systems Integration Officer must validate that the provided topographical heuristic map is mathematically safe. In this lab, we will write a script to automatically check if a heuristic is Admissible and Consistent.

**Steps Performed:**
* **Define a state space graph** with exact path costs and true goal distances ($h^*$).

* **Evaluate** "Heuristic Alpha" and check for violations.

* **Observe** how a failure in the triangle inequality flags a heuristic as unsafe for Graph Search.


```{mermaid}
graph LR
    S((Start<br>h* = 7<br>h = 6))
    A((Node A<br>h* = 5<br>h = 5))
    B((Node B<br>h* = 2<br>h = 1))
    C((Node C<br>h* = 4<br>h = 1))
    G((Goal<br>h* = 0<br>h = 0))

    S -- "Cost: 2" --> A
    S -- "Cost: 5" --> B
    A -- "Cost: 1" --> C
    B -- "Cost: 2" --> G
    C -- "Cost: 4" --> G

    %% Highlight the inconsistent arc
    style A fill:transparent,stroke:#e74c3c,stroke-width:3px
    style C fill:transparent,stroke:#e74c3c,stroke-width:3px
    linkStyle 2 stroke:#e74c3c,stroke-width:3px
```

In [1]:

# Map: {Node: {'neighbors': {Neighbor: Cost}, 'true_cost': h*, 'h_alpha': heuristic}}
tactical_map = {
    'Start': {'neighbors': {'A': 2, 'B': 5}, 'true_cost': 7,  'h_alpha': 6},
    'A':     {'neighbors': {'C': 1},         'true_cost': 5,  'h_alpha': 5},
    'B':     {'neighbors': {'Goal': 2},      'true_cost': 2,  'h_alpha': 1},
    'C':     {'neighbors': {'Goal': 4},      'true_cost': 4,  'h_alpha': 1}, # SUSPICIOUS DROP
    'Goal':  {'neighbors': {},               'true_cost': 0,  'h_alpha': 0}
}

def validate_heuristic(graph):
    is_admissible = True
    is_consistent = True

    for node, data in graph.items():
        h_val = data['h_alpha']
        true_h = data['true_cost']
        
        # 1. Check Admissibility: h(n) <= h*(n)
        if h_val > true_h:
            print(f"[FAIL] Admissibility at {node}: {h_val} overestimates true cost {true_h}")
            is_admissible = False
            
        # 2. Check Consistency: h(A) <= cost(A->C) + h(C)
        for neighbor, arc_cost in data['neighbors'].items():
            neighbor_h = graph[neighbor]['h_alpha']
            if h_val > (arc_cost + neighbor_h):
                print(f"[FAIL] Consistency from {node} -> {neighbor}: "
                      f"{h_val} > ({arc_cost} + {neighbor_h})")
                is_consistent = False

    if is_admissible and is_consistent:
        print("RESULT: Heuristic is SAFE for A* Graph Search.")
    else:
        print("RESULT: Heuristic is UNSAFE.")

print("--- Evaluating Heuristic Alpha ---")
validate_heuristic(tactical_map)

--- Evaluating Heuristic Alpha ---
[FAIL] Consistency from A -> C: 5 > (1 + 1)
RESULT: Heuristic is UNSAFE.


## Interpreting the Results

```{figure} ../../figures/admissibility_consistency.png
---
width: 100%
align: center
name: admissibility_consistency
---
Example of checking admissiabitliy and consistency for $A^*$.
```



When you run this script, the system will flag a Consistency failure from Waypoint A to Waypoint C.
At Waypoint A, the heuristic estimates 5 fuel left to the goal. The agent flies to Waypoint C (burning 1 fuel). However, Waypoint C's heuristic claims there is only 1 fuel left to the goal. Mathematically, $5 \le (1 + 1)$ is false. Because the estimated cost shrank faster than the actual fuel burned, the heuristic is inconsistent. If deployed in a Graph Search, the AI might accidentally lock in a suboptimal path.
